In [1]:
"""
Cricket Batting Analysis Pipeline
Compares two batsmen using YOLO detection, 3D pose estimation, and shot classification
Optimized for 20-frame TCN model with proper joint normalization
"""
!pip install ultralytics

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import tensorflow_hub as hub
from pathlib import Path
import pandas as pd
import time

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.4 MB/s eta 0:00:00


2026-08-03 17:03:43.245227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785776623.460300      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785776623.521005      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785776624.001226      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785776624.001268      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785776624.001271      57 computation_placer.cc:177] computation placer alr

<h1> Load Necessery Models </h1>

In [2]:
class ModelManager:
    """Singleton class to load and manage all models"""
    _instance = None
    _models_loaded = False
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ModelManager, cls).__new__(cls)
        return cls._instance
    
    def __init__(self):
        if not ModelManager._models_loaded:
            self.yolo_model = None
            self.metrabs_model = None
            self.shot_model = None
            ModelManager._models_loaded = True
    
    def load_yolo(self, model_path):
        """Load YOLO model once"""
        if self.yolo_model is None:
            print("Loading YOLO model...")
            from ultralytics import YOLO
            self.yolo_model = YOLO(model_path)
            print("✓ YOLO model loaded!")
        else:
            print("✓ Using cached YOLO model")
        return self.yolo_model
    
    def load_metrabs(self, model_path):
        """Load METRAbs model once"""
        if self.metrabs_model is None:
            print("Loading METRAbs model...")
            self.metrabs_model = hub.load(model_path)
            print("METRAbs model loaded!")
        else:
            print("Using cached METRAbs model")
        return self.metrabs_model
    
    def load_shot_classifier(self, model_path):
        """Load shot classification model once"""
        if self.shot_model is None:
            print("⏳ Loading shot classification model...")
            self.shot_model = keras.models.load_model(model_path)
            print("✓ Shot classifier loaded!")
        else:
            print("✓ Using cached shot classifier")
        return self.shot_model

<h1>Batsman Detection & Frame extraction</h1>

In [3]:
def extract_batsman_frames(video_path, yolo_model, confidence_threshold=0.5):
    """
    Extract frames containing striker batsman using YOLO detection
    Crops to bounding box region only (rest is black)
    
    Args:
        video_path: Path to input video
        yolo_model: Loaded YOLO model
        confidence_threshold: Minimum confidence for detection
    
    Returns:
        List of cropped frames, list of bounding boxes, frame indices
    """
    cap = cv2.VideoCapture(video_path)
    batsman_frames = []
    bounding_boxes = []
    frame_indices = []
    frame_count = 0
    
    print(f"Processing video: {video_path}")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run YOLO detection
        results = yolo_model(frame, verbose=False)
        
        # Check if batsman detected with sufficient confidence
        for result in results:
            if len(result.boxes) > 0:
                confidences = result.boxes.conf.cpu().numpy()
                max_conf_idx = np.argmax(confidences)
                
                if confidences[max_conf_idx] >= confidence_threshold:
                    # Get bounding box in xyxy format
                    box_xyxy = result.boxes.xyxy[max_conf_idx].cpu().numpy()
                    
                    # Convert to integers for cropping
                    x1, y1, x2, y2 = box_xyxy.astype(int)
                    h, w = frame.shape[:2]
                    
                    # Ensure box is within frame bounds
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)
                    
                    # Create black image with same dimensions as original frame
                    masked_frame = np.zeros_like(frame)
                    
                    # Copy only the bounding box region
                    masked_frame[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
                    
                    # Normalized box for METRAbs (if needed)
                    box_normalized = [
                        x1 / w,  # x1
                        y1 / h,  # y1
                        x2 / w,  # x2
                        y2 / h   # y2
                    ]
                    
                    batsman_frames.append(masked_frame)
                    bounding_boxes.append(box_normalized)
                    frame_indices.append(frame_count)
        
        frame_count += 1
    
    cap.release()
    print(f"Extracted {len(batsman_frames)} frames with batsman detected")
    
    return batsman_frames, bounding_boxes, frame_indices



<h1>Image Preprocessing</h1>

In [4]:
def apply_high_boost_filter(image, kernel_size=3, amplification=1.5):
    """
    Apply high boost filter to enhance edges
    High Boost = Original + Amplification * (Original - Blurred)
    """
    img_float = image.astype(np.float32)
    blurred = cv2.GaussianBlur(img_float, (kernel_size, kernel_size), 0)
    high_boost = img_float + amplification * (img_float - blurred)
    high_boost = np.clip(high_boost, 0, 255).astype(np.uint8)
    return high_boost


def preprocess_frames(frames, target_height=480):
    """Resize frames to fixed height and apply high boost filter"""
    preprocessed = []
    
    for frame in frames:
        # Calculate aspect ratio and resize
        h, w = frame.shape[:2]
        aspect_ratio = w / h
        target_width = int(target_height * aspect_ratio)
        
        resized = cv2.resize(frame, (target_width, target_height))
        
        # Apply high boost filter
        filtered = apply_high_boost_filter(resized, kernel_size=3, amplification=1.5)
        
        preprocessed.append(filtered)
    
    return preprocessed

<h1>3d Pose Estimation</h1>

In [5]:
def estimate_3d_poses(frames, bounding_boxes, metrabs_model, 
                     skeleton_type='smpl+head_30', confidence_threshold=0.3):
    """
    Estimate 3D poses for all frames using METRAbs detect_poses
    
    Returns:
        Array of 3D pose coordinates (shape: [num_frames, num_joints, 3])
    """
    poses_3d = []
    
    print("Estimating 3D poses...")
    
    for i, frame in enumerate(frames):
        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Convert to tensor
        img_tensor = tf.convert_to_tensor(rgb_frame)
        
        try:
            # Use detect_poses which automatically detects people
            pred = metrabs_model.detect_poses(img_tensor, skeleton=skeleton_type)
            
            # Get boxes and filter by confidence
            boxes = pred.get('boxes', None)
            
            if boxes is not None:
                conf = boxes[:, 4].numpy()
                mask = conf >= confidence_threshold
                poses3d = pred['poses3d'].numpy()[mask]
            else:
                poses3d = pred['poses3d'].numpy()
            
            # Take the first person (highest confidence, should be the batsman)
            if len(poses3d) > 0:
                poses_3d.append(poses3d[0])
            else:
                # No pose detected, use zeros
                poses_3d.append(np.zeros((30, 3)))
                
        except Exception as e:
            print(f"⚠ Error frame {i}: {e}")
            poses_3d.append(np.zeros((30, 3)))
        
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(frames)} frames")
    
    return np.array(poses_3d)


#POSE NORMALIZATION (MATCHING TRAINING PIPELINE)

def normalize_pose_sequence(pose_sequence):
    """
    Normalize pose sequence exactly like training pipeline
    
    Args:
        pose_sequence: Array of shape (num_frames, num_joints, 3)
    
    Returns:
        Normalized pose sequence
    """
    # Use pelvis (joint 0) as reference point
    pelvis = pose_sequence[:, 0:1, :]  # Shape: (frames, 1, 3)
    centered = pose_sequence - pelvis
    
    # Calculate normalization scale using torso height
    # Distance between pelvis (0) and neck (12)
    if pose_sequence.shape[1] > 12:
        torso_heights = np.linalg.norm(
            pose_sequence[:, 12, :] - pose_sequence[:, 0, :], 
            axis=1
        )
        avg_height = np.mean(torso_heights) + 1e-6
    else:
        # Fallback: use standard deviation
        avg_height = np.std(centered) + 1e-6
    
    normalized = centered / avg_height
    return normalized


<h1>Frame Selection</h1>

In [6]:
def calculate_joint_rmse(pose1, pose2):

    return np.sqrt(np.mean((pose1 - pose2) ** 2))


def select_20_frames_rmse_interval(poses_3d, num_frames=20):
    """
    Select exactly 20 frames using interval-based sampling with RMSE distance
    
    This method:
    1. Divides the sequence into intervals
    2. Within each interval, selects the frame with maximum RMSE from previous selected frame
    
    Args:
        poses_3d: Array of shape (total_frames, num_joints, 3)
        num_frames: Number of frames to select (default: 20)
    
    Returns:
        Selected poses (20, num_joints, 3) and their indices
    """
    total_frames = len(poses_3d)
    
    # If we have fewer frames than needed, interpolate
    if total_frames < num_frames:
        print(f"  Warning: Only {total_frames} frames available, interpolating to {num_frames}")
        indices_old = np.arange(total_frames)
        indices_new = np.linspace(0, total_frames - 1, num_frames)
        
        # Interpolate each joint and coordinate
        interpolated = np.zeros((num_frames, poses_3d.shape[1], poses_3d.shape[2]))
        for j in range(poses_3d.shape[1]):  # For each joint
            for c in range(poses_3d.shape[2]):  # For each coordinate
                interpolated[:, j, c] = np.interp(indices_new, indices_old, poses_3d[:, j, c])
        
        return interpolated, np.arange(num_frames)
    
    # Calculate interval size
    interval_size = total_frames // num_frames
    selected_indices = []
    selected_poses = []
    
    # Always select the first frame
    selected_indices.append(0)
    selected_poses.append(poses_3d[0])
    last_selected_pose = poses_3d[0]
    
    print(f"  Selecting {num_frames} frames from {total_frames} using RMSE interval method")
    print(f"  Interval size: {interval_size}")
    
    # For each subsequent interval
    for i in range(1, num_frames):
        # Define interval boundaries
        interval_start = i * interval_size
        interval_end = min((i + 1) * interval_size, total_frames)
        
        # If this is the last interval, extend to the end
        if i == num_frames - 1:
            interval_end = total_frames
        
        # Calculate RMSE for all frames in this interval compared to last selected frame
        rmse_scores = []
        for frame_idx in range(interval_start, interval_end):
            rmse = calculate_joint_rmse(last_selected_pose, poses_3d[frame_idx])
            rmse_scores.append((frame_idx, rmse))
        
        # Select frame with maximum RMSE (most different from last selected)
        if rmse_scores:
            best_frame_idx = max(rmse_scores, key=lambda x: x[1])[0]
            selected_indices.append(best_frame_idx)
            selected_poses.append(poses_3d[best_frame_idx])
            last_selected_pose = poses_3d[best_frame_idx]
            
            if (i + 1) % 5 == 0:
                print(f"  Selected frames {i+1}/{num_frames}")
    
    selected_poses = np.array(selected_poses)
    selected_indices = np.array(selected_indices)
    
    print(f"  ✓ Selected frame indices: {selected_indices.tolist()}")
    
    return selected_poses, selected_indices

<h1>shot classification</h1>

In [7]:
def classify_shot(poses_3d, shot_model):
    """
    Classify batting shot using trained TCN model
    
    Args:
        poses_3d: 3D pose array (shape: [num_frames, num_joints, 3])
        shot_model: Loaded Keras model
    
    Returns:
        Shot name, confidence, and all probabilities
    """
    # Step 1: Select exactly 20 frames using RMSE interval method
    selected_poses, selected_indices = select_20_frames_rmse_interval(poses_3d, num_frames=20)
    print(f"  Selected 20 frames from {len(poses_3d)} total frames")
    
    # Step 2: Normalize poses (matching training pipeline)
    normalized_poses = normalize_pose_sequence(selected_poses)
    print(f"  Normalized poses: shape {normalized_poses.shape}")
    
    # Step 3: Flatten features: (20, 30, 3) -> (20, 90)
    # Each frame has 30 joints * 3 coords = 90 features
    flattened_poses = normalized_poses.reshape(20, -1)
    print(f"  Flattened poses: shape {flattened_poses.shape}")
    
    # Step 4: Add batch dimension: [1, 20, 90]
    input_data = np.expand_dims(flattened_poses, axis=0)
    print(f"  Input shape to model: {input_data.shape}")
    
    # Step 5: Predict
    prediction = shot_model.predict(input_data, verbose=0)
    
    # Step 6: Get predicted class and confidence
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class]
    
    shot_types = ['drive', 'defence', 'flick', 'pull']
    predicted_shot = shot_types[predicted_class]
    
    return predicted_shot, confidence, prediction[0]


<h1> Pipeline </h1>

In [9]:
def run_complete_pipeline(video1_path, video2_path, 
                         yolo_model_path, metrabs_model_path, shot_model_path,
                         output_dir='output'):
    """
    Run complete analysis pipeline with model caching
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    print("=" * 70)
    print("CRICKET BATTING ANALYSIS PIPELINE")
    print("=" * 70)
    
    # Initialize model manager (loads models once)
    model_manager = ModelManager()
    start_time = time.perf_counter()
    # Load all models (cached after first load)
    yolo_model = model_manager.load_yolo(yolo_model_path)
    metrabs_model = model_manager.load_metrabs(metrabs_model_path)
    shot_model = model_manager.load_shot_classifier(shot_model_path)
    end_time = time.perf_counter()
    execution_time = end_time - start_time

    print(f"Loading all models finished in {execution_time:.6f} seconds.")

    start_time = time.perf_counter()
    # Step 1: Extract frames
    print("\n[1/6] Extracting frames with batsman...")
    frames1, boxes1, indices1 = extract_batsman_frames(video1_path, yolo_model)
    frames2, boxes2, indices2 = extract_batsman_frames(video2_path, yolo_model)
    
    # Step 2: Preprocess
    print("\n[2/6] Preprocessing frames...")
    frames1 = preprocess_frames(frames1, target_height=480)
    frames2 = preprocess_frames(frames2, target_height=480)
    
    # Step 3: 3D pose estimation
    print("\n[3/6] Estimating 3D poses...")
    poses1 = estimate_3d_poses(frames1, boxes1, metrabs_model, 
                              skeleton_type='smpl+head_30', confidence_threshold=0.3)
    poses2 = estimate_3d_poses(frames2, boxes2, metrabs_model,
                              skeleton_type='smpl+head_30', confidence_threshold=0.3)
    
    # Step 4: Shot classification
    print("\n[4/6] Classifying shots...")
    shot1_name, shot1_conf, shot1_probs = classify_shot(poses1, shot_model)
    shot2_name, shot2_conf, shot2_probs = classify_shot(poses2, shot_model)
    
    print(f"\nPlayer 1 Shot: {shot1_name.upper()} (confidence: {shot1_conf:.2%})")
    print(f"   Probabilities: {dict(zip(['drive', 'defence', 'flick', 'pull'], shot1_probs))}")
    print(f"\nPlayer 2 Shot: {shot2_name.upper()} (confidence: {shot2_conf:.2%})")
    print(f"   Probabilities: {dict(zip(['drive', 'defence', 'flick', 'pull'], shot2_probs))}")
    
    # # Step 5: Visualization and analysis
    # print("\n[5/6] Creating visualizations...")
    
    # # Align frame counts for comparison
    # min_frames = min(len(frames1), len(frames2))
    
    # # Create comparison video
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(str(output_path / 'comparison.mp4'), fourcc, 10, 
    #                      (frames1[0].shape[1] * 2, frames1[0].shape[0]))
    
    all_differences = []
    
    end_time = time.perf_counter()
    execution_time = end_time - start_time

    print(f"Execution time taken: {execution_time:.6f} seconds.")
    return {
        'frames1': frames1,
        'frames2': frames2,
        'poses1': poses1,
        'poses2': poses2,
        'shot1': {'name': shot1_name, 'confidence': shot1_conf, 'probabilities': shot1_probs},
        'shot2': {'name': shot2_name, 'confidence': shot2_conf, 'probabilities': shot2_probs},
        'differences': all_differences
    }


In [11]:
if __name__ == "__main__":
    # Define paths
    VIDEO1 = "/kaggle/input/datasets/sourav32/testvideo/Rohit Sharmas BRUTAL Pull Shot Masterclass  Epic Sixes Compilation.mp4"
    VIDEO2 = "/kaggle/input/datasets/sourav32/testvideo/17.mp4"  
    YOLO_MODEL = "/kaggle/input/datasets/sourav32/striker-batsman-detect/batsman_detect.pt"
    METRABS_MODEL = "/kaggle/input/datasets/sourav32/metrabs-model"  
    SHOT_MODEL = "/kaggle/input/datasets/sourav32/shotclassifier/tcn_best_model_20frames.keras"
    OUTPUT_DIR = "/kaggle/working/output"
    
    # Run pipeline (models are cached and reused on subsequent runs)
    results = run_complete_pipeline(
        video1_path=VIDEO1,
        video2_path=VIDEO2,
        yolo_model_path=YOLO_MODEL,
        metrabs_model_path=METRABS_MODEL,
        shot_model_path=SHOT_MODEL,
        output_dir=OUTPUT_DIR
    )
    
    # print("\n   Results available in 'results' dictionary")
    # print(f"   - frames1, frames2: Extracted and preprocessed frames")
    # print(f"   - poses1, poses2: 3D pose estimations")
    # print(f"   - shot1, shot2: Shot classifications")
    # print(f"   - differences: Joint difference data")
    
    # To run again with different videos, just call the function again
    # Models will be reused from cache!

CRICKET BATTING ANALYSIS PIPELINE
✓ Using cached YOLO model
Using cached METRAbs model
✓ Using cached shot classifier
Loading all models finished in 0.000023 seconds.

[1/6] Extracting frames with batsman...
Processing video: /kaggle/input/datasets/sourav32/testvideo/Rohit Sharmas BRUTAL Pull Shot Masterclass  Epic Sixes Compilation.mp4
Extracted 46 frames with batsman detected
Processing video: /kaggle/input/datasets/sourav32/testvideo/17.mp4
Extracted 29 frames with batsman detected

[2/6] Preprocessing frames...

[3/6] Estimating 3D poses...
Estimating 3D poses...
Processed 10/46 frames
Processed 20/46 frames
Processed 30/46 frames
Processed 40/46 frames
Estimating 3D poses...
Processed 10/29 frames
Processed 20/29 frames

[4/6] Classifying shots...
  Selecting 20 frames from 46 using RMSE interval method
  Interval size: 2
  Selected frames 5/20
  Selected frames 10/20
  Selected frames 15/20
  Selected frames 20/20
  ✓ Selected frame indices: [0, 3, 5, 7, 9, 11, 12, 15, 17, 19, 21